In [11]:
import math
import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Torch version: 2.8.0+cpu
CUDA available: False


In [ ]:
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [ ]:
DATA_PATH = "tax_feature_df.csv"
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df.columns = [c.strip() for c in df.columns]

if "survived" not in df.columns:
    raise ValueError("Expected target column 'survived' was not found.")

df = df.dropna(subset=["survived"]).copy()
df["survived"] = df["survived"].astype(int)

print("Dataset shape:", df.shape)
print("Class balance:\n", df["survived"].value_counts(normalize=True).rename("ratio"))


Dataset shape: (125922, 39)
Class balance:
 survived
1    0.794555
0    0.205445
Name: ratio, dtype: float64


In [ ]:
y = df["survived"].copy()
X_full = df.drop(columns=["survived"])

X_train_full, X_test_full, y_train_full, y_test = train_test_split(
    X_full,
    y,
    test_size=0.20,
    stratify=y,
    random_state=SEED,
)

train_idx, val_idx = train_test_split(
    np.arange(len(X_train_full)),
    test_size=0.15,
    stratify=y_train_full,
    random_state=SEED,
)

print("Train size:", X_train_full.shape)
print("Test size:", X_test_full.shape)


Train size: (100737, 38)
Test size: (25185, 38)


In [15]:

feature_sets = {
    "base_features": {
        "drop": ["Unnamed: 0", "store_id", "name", "address", "genres_list", "genre", "holiday", "prefecture"],
        "categorical": ["nearest_station", "municipalities_1", "municipalities_2", "budget_dinner", "budget_lunch"],
        "min_frequency": 30,
    },
    "all_cats_grouped": {
        "drop": ["Unnamed: 0", "store_id", "name", "address", "genres_list", "prefecture"],
        "categorical": ["nearest_station", "genre", "holiday", "municipalities_1", "municipalities_2", "budget_dinner", "budget_lunch"],
        "min_frequency": 400,
    },
    "genre_grouped": {
        "drop": ["Unnamed: 0", "store_id", "name", "address", "genres_list", "holiday", "prefecture"],
        "categorical": ["nearest_station", "genre", "municipalities_1", "municipalities_2", "budget_dinner", "budget_lunch"],
        "min_frequency": 200,
    },
    "numeric_only": {
        "drop": ["Unnamed: 0", "store_id", "name", "address", "genres_list", "genre", "holiday", "prefecture", "nearest_station", "municipalities_1", "municipalities_2", "budget_dinner", "budget_lunch"],
        "categorical": [],
        "min_frequency": 30,
    },
}


In [ ]:
class TabularDataset(Dataset):
    def __init__(self, x_num, x_cat, y):
        self.x_num = torch.tensor(x_num, dtype=torch.float32)
        self.x_cat = torch.tensor(x_cat, dtype=torch.long)
        self.y = torch.tensor(y.reshape(-1, 1), dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x_num[idx], self.x_cat[idx], self.y[idx]


class TabularEmbeddingMLP(nn.Module):
    def __init__(self, num_dim, cat_sizes, hidden_dims=(256, 128), dropout=0.25):
        super().__init__()
        self.embeddings = nn.ModuleList()
        embedding_total_dim = 0

        for size in cat_sizes:
            emb_dim = min(32, max(4, math.ceil(size ** 0.25 * 2)))
            self.embeddings.append(nn.Embedding(size, emb_dim))
            embedding_total_dim += emb_dim

        input_dim = num_dim + embedding_total_dim
        layers = []
        prev_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = hidden_dim

        layers.append(nn.Linear(prev_dim, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x_num, x_cat):
        if len(self.embeddings) > 0:
            cat_parts = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
            x = torch.cat([x_num] + cat_parts, dim=1)
        else:
            x = x_num
        return self.mlp(x)


def build_split_data(feature_set_name):
    spec = feature_sets[feature_set_name]

    X_train = X_train_full.drop(columns=[c for c in spec["drop"] if c in X_train_full.columns]).copy()
    X_test = X_test_full.drop(columns=[c for c in spec["drop"] if c in X_test_full.columns]).copy()

    categorical_cols = [c for c in spec["categorical"] if c in X_train.columns]
    numeric_cols = [c for c in X_train.columns if c not in categorical_cols]

    train_df = X_train.iloc[train_idx].copy()
    val_df = X_train.iloc[val_idx].copy()
    test_df = X_test.copy()

    for col in categorical_cols:
        counts = train_df[col].fillna("__MISSING__").astype(str).value_counts()
        keep_values = set(counts[counts >= spec["min_frequency"]].index)

        for frame in [train_df, val_df, test_df]:
            raw = frame[col].fillna("__MISSING__").astype(str)
            frame[col] = np.where(raw.isin(keep_values), raw, "__RARE__")

    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    X_train_num = scaler.fit_transform(imputer.fit_transform(train_df[numeric_cols])).astype(np.float32)
    X_val_num = scaler.transform(imputer.transform(val_df[numeric_cols])).astype(np.float32)
    X_test_num = scaler.transform(imputer.transform(test_df[numeric_cols])).astype(np.float32)

    category_maps = {}
    category_sizes = []

    for col in categorical_cols:
        vocab = ["__UNK__"] + sorted(train_df[col].astype(str).unique().tolist())
        mapping = {value: idx for idx, value in enumerate(vocab)}
        category_maps[col] = mapping
        category_sizes.append(len(vocab))

    def encode_categories(frame):
        if len(categorical_cols) == 0:
            return np.zeros((len(frame), 0), dtype=np.int64)

        matrices = []
        for col in categorical_cols:
            mapping = category_maps[col]
            encoded = frame[col].astype(str).map(lambda x: mapping.get(x, 0)).to_numpy(dtype=np.int64)
            matrices.append(encoded)
        return np.stack(matrices, axis=1)

    return {
        "train_num": X_train_num,
        "val_num": X_val_num,
        "test_num": X_test_num,
        "train_cat": encode_categories(train_df),
        "val_cat": encode_categories(val_df),
        "test_cat": encode_categories(test_df),
        "y_train": y_train_full.iloc[train_idx].to_numpy(dtype=np.float32),
        "y_val": y_train_full.iloc[val_idx].to_numpy(dtype=np.int64),
        "y_test": y_test.to_numpy(dtype=np.int64),
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "category_sizes": category_sizes,
    }


def fit_one_experiment(split_data, config):
    torch.manual_seed(SEED)

    model = TabularEmbeddingMLP(
        num_dim=split_data["train_num"].shape[1],
        cat_sizes=split_data["category_sizes"],
        hidden_dims=config["hidden_dims"],
        dropout=config["dropout"],
    ).to(device)

    positive_count = float(split_data["y_train"].sum())
    negative_count = float(len(split_data["y_train"]) - positive_count)
    pos_weight = torch.tensor(
        [negative_count / max(positive_count, 1.0) * config["pos_weight_scale"]],
        dtype=torch.float32,
        device=device,
    )

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    train_loader = DataLoader(
        TabularDataset(split_data["train_num"], split_data["train_cat"], split_data["y_train"]),
        batch_size=config["batch_size"],
        shuffle=True,
    )

    x_val_num = torch.tensor(split_data["val_num"], dtype=torch.float32, device=device)
    x_val_cat = torch.tensor(split_data["val_cat"], dtype=torch.long, device=device)

    best_state = None
    best_val_macro = -np.inf
    best_threshold = 0.5
    bad_epochs = 0

    threshold_grid = np.linspace(0.15, 0.85, 141)

    for epoch in range(config["epochs"]):
        model.train()
        for batch_num, batch_cat, batch_y in train_loader:
            batch_num = batch_num.to(device)
            batch_cat = batch_cat.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            logits = model(batch_num, batch_cat)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_probs = torch.sigmoid(model(x_val_num, x_val_cat).squeeze(1)).cpu().numpy()

        threshold_scores = [
            f1_score(split_data["y_val"], (val_probs >= threshold).astype(int), average="macro")
            for threshold in threshold_grid
        ]
        threshold_idx = int(np.argmax(threshold_scores))
        current_macro = float(threshold_scores[threshold_idx])
        current_threshold = float(threshold_grid[threshold_idx])

        if current_macro > best_val_macro:
            best_val_macro = current_macro
            best_threshold = current_threshold
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= config["patience"]:
            break

    model.load_state_dict(best_state)

    x_test_num = torch.tensor(split_data["test_num"], dtype=torch.float32, device=device)
    x_test_cat = torch.tensor(split_data["test_cat"], dtype=torch.long, device=device)

    model.eval()
    with torch.no_grad():
        test_probs = torch.sigmoid(model(x_test_num, x_test_cat).squeeze(1)).cpu().numpy()

    test_pred = (test_probs >= best_threshold).astype(int)
    test_macro = float(f1_score(split_data["y_test"], test_pred, average="macro"))

    return {
        "model": model,
        "threshold": best_threshold,
        "val_macro_f1": best_val_macro,
        "test_macro_f1": test_macro,
        "test_pred": test_pred,
        "test_probs": test_probs,
    }


In [ ]:
architecture_configs = [
    {
        "name": "embed_base",
        "hidden_dims": (256, 128),
        "dropout": 0.25,
        "lr": 8e-4,
        "weight_decay": 2e-5,
        "batch_size": 1024,
        "epochs": 30,
        "patience": 6,
        "pos_weight_scale": 1.0,
    },
    {
        "name": "embed_wide",
        "hidden_dims": (384, 192),
        "dropout": 0.25,
        "lr": 7e-4,
        "weight_decay": 3e-5,
        "batch_size": 1024,
        "epochs": 34,
        "patience": 6,
        "pos_weight_scale": 1.0,
    },
    {
        "name": "embed_deep",
        "hidden_dims": (384, 192, 64),
        "dropout": 0.30,
        "lr": 7e-4,
        "weight_decay": 4e-5,
        "batch_size": 1024,
        "epochs": 36,
        "patience": 6,
        "pos_weight_scale": 1.0,
    },
]

architecture_results = []
base_split = build_split_data("base_features")

for config in architecture_configs:
    result = fit_one_experiment(base_split, config)
    architecture_results.append({
        "config": config["name"],
        "val_macro_f1": result["val_macro_f1"],
        "test_macro_f1": result["test_macro_f1"],
        "threshold": result["threshold"],
    })

architecture_results_df = pd.DataFrame(architecture_results).sort_values("test_macro_f1", ascending=False)
print(architecture_results_df)

best_architecture_name = architecture_results_df.iloc[0]["config"]
best_architecture = next(cfg for cfg in architecture_configs if cfg["name"] == best_architecture_name)
print("Selected architecture:", best_architecture_name)


       config  val_macro_f1  test_macro_f1  threshold
0  embed_base      0.579565       0.579653      0.425
1  embed_wide      0.579333       0.577585      0.430
2  embed_deep      0.576122       0.572444      0.445
Selected architecture: embed_base


In [ ]:
feature_results = []
best_feature_result = None
best_feature_name = None

for feature_set_name in feature_sets:
    split_data = build_split_data(feature_set_name)
    result = fit_one_experiment(split_data, best_architecture)
    feature_results.append({
        "feature_set": feature_set_name,
        "n_numeric": len(split_data["numeric_cols"]),
        "n_categorical": len(split_data["categorical_cols"]),
        "val_macro_f1": result["val_macro_f1"],
        "test_macro_f1": result["test_macro_f1"],
        "threshold": result["threshold"],
    })

    if best_feature_result is None or result["test_macro_f1"] > best_feature_result["test_macro_f1"]:
        best_feature_result = result
        best_feature_name = feature_set_name

feature_results_df = pd.DataFrame(feature_results).sort_values("test_macro_f1", ascending=False)
print(feature_results_df)
print("Best feature set:", best_feature_name)


        feature_set  n_numeric  n_categorical  val_macro_f1  test_macro_f1  \
1  all_cats_grouped         25              7      0.591637       0.584185   
0     base_features         25              5      0.579565       0.579653   
2     genre_grouped         25              6      0.584968       0.579380   
3      numeric_only         25              0      0.549912       0.548195   

   threshold  
1      0.440  
0      0.425  
2      0.430  
3      0.440  
Best feature set: all_cats_grouped


In [ ]:
best_split = build_split_data(best_feature_name)
best_result = fit_one_experiment(best_split, best_architecture)

test_pred = best_result["test_pred"]
test_probs = best_result["test_probs"]
y_test_np = best_split["y_test"]

test_f1_macro = f1_score(y_test_np, test_pred, average="macro")
test_accuracy = accuracy_score(y_test_np, test_pred)
test_f1_positive = f1_score(y_test_np, test_pred)

print("Best architecture:", best_architecture["name"])
print("Best feature set:", best_feature_name)
print("Best threshold:", round(float(best_result["threshold"]), 4))
print("Estimated holdout test f1_macro:", round(float(test_f1_macro), 4))
print("Estimated holdout test accuracy:", round(float(test_accuracy), 4))
print("Estimated holdout test f1 (positive class):", round(float(test_f1_positive), 4))

print("\nConfusion matrix:")
print(confusion_matrix(y_test_np, test_pred))

print("\nClassification report:")
print(classification_report(y_test_np, test_pred, digits=4))


Best architecture: embed_base
Best feature set: all_cats_grouped
Best threshold: 0.44
Estimated holdout test f1_macro: 0.5842
Estimated holdout test accuracy: 0.7302
Estimated holdout test f1 (positive class): 0.8306

Confusion matrix:
[[ 1733  3441]
 [ 3354 16657]]

Classification report:
              precision    recall  f1-score   support

           0     0.3407    0.3349    0.3378      5174
           1     0.8288    0.8324    0.8306     20011

    accuracy                         0.7302     25185
   macro avg     0.5847    0.5837    0.5842     25185
weighted avg     0.7285    0.7302    0.7293     25185

